# Introduction to Agentic AI — What Are Agents and Why Do They Matter?

## What This Notebook Teaches You

Most AI tutorials show you how to **call an LLM**. This one shows you how to build **systems that think, act, and learn autonomously**.

By the end of this notebook, you will understand:

1. **What makes a system "agentic"** — the formal definition and the practical spectrum
2. **The agentic spectrum (L0–L4)** — from single prompts to multi-agent collaboration
3. **The history of AI agents** — from expert systems to LLM-based agents
4. **The perception-action loop** — the universal pattern underlying all agents
5. **When agents add value** — and when simpler approaches win

Every concept includes **runnable code** demonstrating the difference between agentic and non-agentic systems.

---

> **Prerequisites**: Complete the [prompt-engineering/](../prompt-engineering/) module (especially intro, CoT, task decomposition, prompt chaining).
> **Runtime**: Google Colab with T4 GPU (free tier works).
> **Time**: ~45–60 minutes.

## Part 0: Environment Setup

We load **Qwen3-14B**, the same open-source model used throughout the prompt-engineering and RAG modules. 4-bit quantization fits it on a free Colab T4 GPU.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. What is an AI Agent?

### 1.1 — The Formal Definition

Russell and Norvig's classic AI textbook (*Artificial Intelligence: A Modern Approach*) defines an **agent** as:

> *"An agent is anything that can be viewed as perceiving its environment through sensors and acting upon that environment through actuators."*

This definition has four components:

| Component | Classical AI | LLM-Based Agent |
|-----------|-------------|-----------------|
| **Perception** | Camera, microphone, sensors | User input, tool results, API responses |
| **Reasoning** | Rule engine, search, planning | LLM inference (next-token prediction) |
| **Action** | Motor, speaker, actuator | Tool calls, API requests, text output |
| **Environment** | Physical world, game board | Digital workspace, databases, web |

### 1.2 — What Makes an LLM System "Agentic"?

A raw LLM is **not** an agent. It is a stateless function: text in, text out. It has no:
- **Memory** — each call is independent (no state between calls)
- **Goals** — it does not "want" anything; it predicts likely next tokens
- **Tools** — it can only generate text, not act on the world
- **Autonomy** — it responds to exactly one prompt, then stops

An LLM becomes agentic when we wrap it in a **loop with state, tools, and goals**:

```
+-----------------------------------------------+
|              AGENT SYSTEM                      |
|                                                |
|   [Memory/State] --> [LLM] --> [Tools/Actions] |
|        ^                            |          |
|        |         [Loop Control]     |          |
|        +----------------------------+          |
+-----------------------------------------------+
```

The key insight: **agency comes from the loop, not the model.** The same LLM becomes an agent when placed inside a system that gives it memory, tools, and iterative control.

## 2. The Agentic Spectrum — Five Levels of Autonomy

Not all AI systems are equally "agentic." Think of it as a spectrum:

| Level | Name | Description | Example |
|-------|------|-------------|---------|
| **L0** | Single Call | One prompt, one response | "Translate this to French" |
| **L1** | Prompt Chain | Output of call N feeds into call N+1 | Summarize, then extract key points |
| **L2** | Tool-Augmented | LLM can call external functions | Calculator, web search, database |
| **L3** | Autonomous Agent | LLM decides what to do next in a loop | ReAct agent solving research questions |
| **L4** | Multi-Agent System | Multiple agents collaborating | Researcher + Writer + Editor team |

Each level adds **capability and complexity**. The key engineering question is always: *what is the minimum level needed for this task?*

Let us see each level in action on the **same task**:

> **Task**: *"What is 47 times 83, and is the result a prime number?"*

This requires: (1) multiplication, (2) primality testing — things LLMs notoriously get wrong.

### 2.1 — Level 0: Single LLM Call

The simplest approach: just ask the question directly.

In [ ]:
# L0: Single LLM Call — just ask directly
response_l0 = generate([
    {"role": "user", "content": "What is 47 times 83, and is the result a prime number? Give the final answer."}
], temperature=0.1)

print("L0 (Single Call) Response:")
print(response_l0)
print()

# Ground truth
print("--- Ground Truth ---")
product = 47 * 83
print(f"47 x 83 = {product}")
is_prime = all(product % i != 0 for i in range(2, int(product**0.5) + 1))
print(f"{product} is prime: {is_prime} (it equals 47 x 83, so it is composite)")

### 2.2 — Level 1: Prompt Chain

Break the task into steps and chain them. Each call handles one sub-task.

In [ ]:
# L1: Prompt Chain — break into sequential steps

# Step 1: Calculate
step1 = generate([
    {"role": "user", "content": "Calculate 47 multiplied by 83. Return ONLY the number."}
], temperature=0.1, max_new_tokens=50)
print(f"Step 1 (Calculate): {step1.strip()}")

# Step 2: Check primality
step2 = generate([
    {"role": "user", "content": f"Is {step1.strip()} a prime number? Think step by step, then answer yes or no."}
], temperature=0.1)
print(f"\nStep 2 (Prime check):\n{step2}")
print(f"\nL1 chain used 2 separate LLM calls.")

### 2.3 — Level 2: Tool-Augmented LLM

Give the LLM access to actual tools. It can now compute rather than guess.

In [ ]:
# L2: Tool-Augmented — LLM has access to real tools

def calculator(expression: str) -> str:
    """Safely evaluate a math expression."""
    try:
        allowed = set('0123456789+-*/.() ')
        if not all(c in allowed for c in expression):
            return f"Error: invalid characters in expression"
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

def is_prime_tool(n: str) -> str:
    """Check if a number is prime."""
    try:
        num = int(n)
        if num < 2:
            return f"{num} is not prime"
        for i in range(2, int(num**0.5) + 1):
            if num % i == 0:
                return f"{num} is NOT prime (divisible by {i}: {num} = {i} x {num // i})"
        return f"{num} IS prime"
    except:
        return f"Error: {n} is not a valid integer"

# Demonstrate the tools
calc_result = calculator("47 * 83")
prime_result = is_prime_tool(calc_result)

print("L2 (Tool-Augmented):")
print(f"  calculator('47 * 83') -> {calc_result}")
print(f"  is_prime_tool('{calc_result}') -> {prime_result}")
print(f"\nTools give guaranteed correct answers.")

### 2.4 — Level 3: Autonomous Agent Loop

The LLM decides **on its own** which tools to call and when to stop. This is the core agentic pattern.

In [ ]:
# L3: Autonomous Agent — LLM decides what to do in a loop

def simple_agent(task, max_steps=5):
    """The simplest possible agent: a loop where the LLM decides what to do."""

    tools_desc = """Available tools:
- calculator(expression): Evaluate a math expression. Example: calculator(47 * 83)
- is_prime(n): Check if a number is prime. Example: is_prime(3901)

Respond with EXACTLY one of:
  TOOL: tool_name(argument)    -- to use a tool
  ANSWER: your final answer    -- when you are done"""

    messages = [
        {"role": "system", "content": f"You are a helpful agent with tools.\n\n{tools_desc}\n\nUse tools for all calculations."},
        {"role": "user", "content": task}
    ]

    tool_funcs = {"calculator": calculator, "is_prime": is_prime_tool}

    print(f"Agent Task: {task}")
    print("=" * 60)

    for step in range(max_steps):
        response = generate(messages, temperature=0.1, max_new_tokens=200)
        print(f"\nStep {step + 1}: {response.strip()[:200]}")

        if "TOOL:" in response:
            tool_line = [l for l in response.split("\n") if "TOOL:" in l][0]
            tool_str = tool_line.split("TOOL:")[1].strip()
            match = re.match(r'(\w+)\((.*)\)', tool_str)
            if match:
                func_name = match.group(1)
                arg = match.group(2).strip().strip('"').strip("'")
                if func_name in tool_funcs:
                    result = tool_funcs[func_name](arg)
                    print(f"  -> Tool result: {result}")
                    messages.append({"role": "assistant", "content": response})
                    messages.append({"role": "user", "content": f"Tool result: {result}"})
                else:
                    messages.append({"role": "assistant", "content": response})
                    messages.append({"role": "user", "content": f"Error: Unknown tool '{func_name}'. Available: calculator, is_prime"})
            continue

        if "ANSWER:" in response:
            answer = response.split("ANSWER:")[1].strip()
            print(f"\n{'=' * 60}")
            print(f"Final Answer: {answer}")
            print(f"Steps taken: {step + 1}")
            return answer

    print("Agent reached max steps.")
    return None

agent_answer = simple_agent("What is 47 times 83, and is the result a prime number?")

### 2.5 — Comparison Across Levels

| Aspect | L0: Single | L1: Chain | L2: Tool-Aug | L3: Agent | L4: Multi-Agent |
|--------|-----------|-----------|-------------|-----------|-----------------|
| **LLM Calls** | 1 | 2-5 | 1-3 | 3-10+ | 10-50+ |
| **Correctness** | Low (guesses) | Medium | High (tools) | High | Highest |
| **Flexibility** | Fixed | Fixed steps | LLM picks tools | LLM decides all | Specialized agents |
| **Cost** | Lowest | Low | Medium | Medium-High | High |
| **Complexity** | None | Simple | Moderate | Significant | Complex |

**Key insight**: use the **minimum level** that solves your task reliably. Do not build a multi-agent system when a prompt chain will do.

## 3. A Brief History of AI Agents

AI agents did not start with LLMs. The concept has evolved through decades of research:

| Era | Approach | Agent Type | Key Idea |
|-----|----------|-----------|----------|
| 1970s-80s | **Expert Systems** | Rule-based | IF-THEN rules, knowledge bases (MYCIN, DENDRAL) |
| 1990s | **BDI Architecture** | Belief-Desire-Intention | Agents with beliefs about the world, desires (goals), and intentions (plans) |
| 2000s | **Multi-Agent Systems** | Cooperative/Competitive | Game theory, auction mechanisms, negotiation protocols |
| 2010s | **RL Agents** | Reward-optimizing | AlphaGo, Atari agents — learn from environment rewards |
| 2022+ | **LLM Agents** | Language-based | Use LLMs as reasoning engine, tools as actuators |

### Key Papers in LLM-Based Agents

| Paper | Authors | Year | Contribution |
|-------|---------|------|-------------|
| *ReAct: Synergizing Reasoning and Acting* | Yao et al. | 2022 | The Thought-Action-Observation loop |
| *Toolformer: LMs Can Teach Themselves to Use Tools* | Schick et al. | 2023 | LLMs learning when and how to use tools |
| *Generative Agents: Interactive Simulacra* | Park et al. | 2023 | Agents with memory, reflection, planning |
| *Reflexion: Language Agents with Verbal RL* | Shinn et al. | 2023 | Agents that learn from their mistakes |
| *Self-Refine: Iterative Refinement* | Madaan et al. | 2023 | Agents that critique and improve their output |
| *A Survey on LLM-based Autonomous Agents* | Wang et al. | 2024 | Comprehensive taxonomy |

The LLM agent revolution happened because **language models became good enough** to serve as general-purpose reasoning engines.

## 4. The Perception-Action Loop

Every agent follows the same fundamental loop:

```
PERCEIVE --> THINK --> ACT --> OBSERVE --> UPDATE STATE --> (repeat or stop)
```

### Mathematical Formulation

At each timestep *t*, the agent:

1. Observes: **o_t** = observe(environment)
2. Updates state: **s_t** = update(s_{t-1}, o_t)
3. Selects action: **a_t** = policy(s_t)
4. Executes: environment = execute(a_t)
5. Checks: if done(s_t) then stop, else t = t+1

For LLM agents:
- **Observation** = tool results, user messages, error messages
- **State** = conversation history + working memory
- **Policy** = the LLM itself (given state, generate next action)
- **Action** = tool call, text response, or "done"

### Environment Types

| Property | Simple | Complex |
|----------|--------|---------|
| Deterministic vs Stochastic | Calculator always returns same result | Web search varies |
| Episodic vs Sequential | Each task independent | Agent learns across tasks |
| Fully vs Partially Observable | Agent sees all info | Must search for info |
| Static vs Dynamic | Environment waits | New info arrives mid-task |

Most LLM agent environments are **stochastic**, **sequential**, **partially observable**, and **static**.

## 5. Building Your First Agent Loop

Let us build the simplest possible agent loop. This agent:
1. Receives a complex question
2. Thinks about it across multiple iterations
3. Decides when it has a good enough answer
4. Tracks its reasoning trace

In [ ]:
@dataclass
class AgentState:
    """Tracks everything about an agent's execution."""
    goal: str
    thoughts: List[str] = field(default_factory=list)
    current_step: int = 0
    max_steps: int = 5
    is_complete: bool = False
    final_answer: Optional[str] = None
    token_estimate: int = 0
    start_time: float = 0.0

    def elapsed(self):
        return time.time() - self.start_time

def thinking_agent(goal: str, max_steps: int = 5) -> AgentState:
    """An agent that thinks iteratively about a question."""
    state = AgentState(goal=goal, max_steps=max_steps, start_time=time.time())

    system_prompt = """You are a thoughtful analytical agent. You think step by step across multiple iterations.

At each step, respond with EXACTLY one of:
  THINKING: [your reasoning for this step]
  FINAL_ANSWER: [your complete answer]

Guidelines:
- Break complex questions into sub-problems
- Each thinking step should address ONE aspect
- Only give FINAL_ANSWER when you have thoroughly analyzed the question"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Question: {goal}"}
    ]

    print(f"Goal: {goal}")
    print("-" * 60)

    for step in range(max_steps):
        state.current_step = step + 1
        response = generate(messages, temperature=0.3, max_new_tokens=300)
        state.token_estimate += len(response.split())

        if "FINAL_ANSWER:" in response:
            answer = response.split("FINAL_ANSWER:")[1].strip()
            state.final_answer = answer
            state.is_complete = True
            print(f"\nStep {step+1} [ANSWER]: {answer[:300]}")
            break
        elif "THINKING:" in response:
            thought = response.split("THINKING:")[1].strip()
            state.thoughts.append(thought)
            print(f"\nStep {step+1} [THINK]: {thought[:200]}")
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": "Continue. THINKING: or FINAL_ANSWER:"})
        else:
            state.thoughts.append(response.strip())
            print(f"\nStep {step+1}: {response.strip()[:200]}")
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": "Use THINKING: or FINAL_ANSWER: format."})

    if not state.is_complete:
        print(f"\nReached max steps ({max_steps}) without final answer.")

    print(f"\n{'-' * 60}")
    print(f"Stats: {state.current_step} steps | ~{state.token_estimate} tokens | {state.elapsed():.1f}s")
    return state

In [ ]:
# Complex multi-faceted question — should take several steps
state1 = thinking_agent(
    "What are the three main differences between classical machine learning "
    "and deep learning, and when would you choose one over the other?",
    max_steps=5
)

In [ ]:
# Simple question — should terminate quickly
state2 = thinking_agent("What is the capital of Japan?", max_steps=5)

In [ ]:
# Analytical question requiring structured thinking
state3 = thinking_agent(
    "A company has 1000 employees and needs to cut costs by 15% without layoffs. "
    "Propose three different strategies and analyze the trade-offs of each.",
    max_steps=6
)

### 5.1 — Analyzing Agent Behavior

Notice the pattern:
- **Simple questions** terminate in 1-2 steps — the loop adds no value.
- **Complex questions** use 3-5 steps, each adding a new dimension of analysis.
- **Open-ended questions** may use all available steps.

This is the fundamental trade-off: **more steps = more thorough but more expensive.**

In [ ]:
# Side-by-side: single call vs agent loop
task = ("What are the three main differences between classical machine learning "
        "and deep learning, and when would you choose one over the other?")

# Single call
t0 = time.time()
single = generate([{"role": "user", "content": task}], temperature=0.3, max_new_tokens=500)
t_single = time.time() - t0

print("=" * 60)
print("SINGLE CALL (L0):")
print("=" * 60)
print(single[:500])
print(f"\nTime: {t_single:.1f}s | Words: {len(single.split())}")

print("\n" + "=" * 60)
print("AGENT LOOP (L3):")
print("=" * 60)
if state1.final_answer:
    print(state1.final_answer[:500])
else:
    for i, t in enumerate(state1.thoughts):
        print(f"  Thought {i+1}: {t[:100]}...")
print(f"\nTime: {state1.elapsed():.1f}s | Steps: {state1.current_step} | ~Tokens: {state1.token_estimate}")

## 6. When Agents Add Value — And When They Don't

| Task Type | Best Approach | Why |
|-----------|--------------|-----|
| Simple factual Q&A | **L0: Single call** | Agent adds cost, not quality |
| Translation, formatting | **L0: Single call** | Well-defined, no iteration needed |
| Multi-step analysis | **L1: Chain** or **L3: Agent** | Benefits from decomposition |
| Tasks needing computation | **L2+: Tools required** | LLMs cannot reliably do math |
| Open-ended research | **L3: Agent** | Needs iterative search and synthesis |
| Complex multi-domain tasks | **L4: Multi-agent** | Requires specialization |

### The 3 Questions Before Building an Agent

1. **Does this task need iteration?** If a single prompt works, do not use an agent.
2. **Does this task need tools?** If computation/search is needed, use at least L2.
3. **Does this task need multiple perspectives?** Consider L4 for specialization.

## 7. The Architecture of Modern LLM Agents

```
+-----------------------------------------------------------+
|                  AGENT ARCHITECTURE                        |
|                                                           |
|  [System Prompt]  [Memory]  [Guardrails]                  |
|       |              |           |                        |
|       v              v           v                        |
|  +------------------------------------------------+      |
|  |              AGENT LOOP                         |      |
|  |  Perceive -> Think -> Act -> Observe -> Update  |      |
|  +------------------------------------------------+      |
|       |                          |                        |
|  [Tool Registry]         [Output Parser]                  |
|  - Calculator            - JSON extraction                |
|  - Search                - Schema validation              |
|  - Code execution        - Retry logic                    |
+-----------------------------------------------------------+
```

### Course Roadmap

| Component | Notebook |
|-----------|----------|
| Agent Loop | **02** the_agent_loop |
| Tool Use | **03** tool_use_and_function_calling |
| Output Parsing | **04** structured_output_parsing |
| ReAct Agent | **05** building_a_react_agent |
| Planning | **06** plan_and_execute |
| Reflection | **07** reflection_and_self_critique |
| Tree of Thought | **08** tree_of_thought |
| Memory (Short) | **10** agent_memory_short_term |
| Memory (Long) | **11** agent_memory_long_term |
| Memory (Graph) | **12** knowledge_graph_memory |
| Safety | **24** agent_safety_and_guardrails |
| Multi-Agent | **17-23** Module 4 |
| Capstone | **32-37** Module 7 |

## 8. Key Takeaways

1. **An agent is a loop, not a model.** The LLM is the reasoning engine; the agent is the system around it.
2. **The agentic spectrum (L0-L4)** helps you pick the right level of complexity.
3. **The perception-action loop is universal** — every agent follows perceive, think, act, observe, update.
4. **LLM agents are recent (2022+)** — key papers: ReAct, Reflexion, Generative Agents, Toolformer.
5. **Agents trade cost for capability** — more autonomy means more LLM calls and tokens.
6. **Infrastructure matters more than the model** — tools, parsing, memory, and guardrails determine quality.

### Next: Notebook 02 — The Agent Loop

We will build a proper `AgentState` class and configurable `AgentLoop` with multiple termination strategies.